# Setup

In [ ]:
import json
import csv
import os
import re
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Hilfsfunktionen

In [ ]:
def read_md_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

def insert_content(assessment_prompt, candidate_profile, job_description):
    return assessment_prompt.replace("{CANDIDATE_PROFILE}", candidate_profile).replace("{JOB_DESCRIPTION}", job_description)

def replace_name(candidate_profile, new_name):
    return candidate_profile.replace("{NAME}", new_name)

def get_next_experiment_dir(model_name):
    base_dir = f"../Results/{model_name}"
    os.makedirs(base_dir, exist_ok=True)

    existing_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()]
    if not existing_dirs:
        next_num = 1
    else:
        next_num = max(int(d) for d in existing_dirs) + 1

    exp_dir = os.path.join(base_dir, str(next_num))
    os.makedirs(exp_dir, exist_ok=True)
    return exp_dir

def save_result(response_text, original_file_name, rank, target_dir, role_folder, try_no, gender):
    base_name = original_file_name.replace(".md", "")
    role_dir = os.path.join(target_dir, role_folder, f"{base_name}_{rank}_{gender}")
    os.makedirs(role_dir, exist_ok=True)

    json_str = response_text
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', response_text, re.DOTALL)
    if match:
        json_str = match.group(1)

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON for {original_file_name} attempt {try_no}: {e}")
        data = {"raw_text": response_text}

    data["file_name"] = original_file_name
    data["rank"] = rank
    data["try"] = try_no
    data["gender"] = gender

    output_filename = f"try_{try_no}.json"
    output_path = os.path.join(role_dir, output_filename)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    return output_path

def get_remote_client():
    load_dotenv()
    api_key = os.getenv("OSKI_API_KEY")
    base_url = os.getenv("OSKI_BASE_URL")

    if not api_key:
        raise ValueError("Missing OSKI_API_KEY in environment or .env file.")
    if not base_url:
        raise ValueError("Missing OSKI_BASE_URL in environment or .env file.")

    return OpenAI(api_key=api_key, base_url=base_url)

def chat_json(client, model_name, prompt, temperature=0.5):
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a research assistant. Return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

### Experiment

In [ ]:
MODEL_NAME = "Mistral Small 3.2 24B Instruct 2506"
RESULT_MODEL_DIR = "OSKI_Mistral_Small_3.2_24B_Instruct_2506"
EXP_DIR = get_next_experiment_dir(RESULT_MODEL_DIR)
NUM_TRIES = 5  # 10
client = get_remote_client()

In [ ]:
names_by_rank = {}

def load_names(filepath, role, col_name):
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rank = int(row["Rang"])

            if rank not in names_by_rank:
                names_by_rank[rank] = {}

            names_by_rank[rank][role] = row[col_name]

base_path = "../namen"

load_names(os.path.join(base_path, "frauen.csv"), "female", col_name="Vorname")
load_names(os.path.join(base_path, "männer.csv"), "male", col_name="Vorname")
load_names(os.path.join(base_path, "nachnamen.csv"), "surname", col_name="Nachname")

In [ ]:
for folder in os.listdir("../CV"):
    for file in os.listdir(os.path.join("../CV", folder)):
        for rank in range(1, 6):
            name_male = names_by_rank[rank].get("male") + " " + names_by_rank[rank].get("surname")
            name_female = names_by_rank[rank].get("female") + " " + names_by_rank[rank].get("surname")
            candidate_profile = read_md_file(os.path.join("../CV", folder, file))
            job_description = read_md_file(os.path.join("../Stellenausschreibungen", folder, f"{folder}.md"))
            assessment_prompt = read_md_file(r"..\prompts\bewertung.txt")

            for try_no in range(1, NUM_TRIES + 1):
                print(f"Processing {file} (Rank {rank}, Try {try_no}), Male...")
                prompt_male = insert_content(assessment_prompt, replace_name(candidate_profile, name_male), job_description)
                response_male_text = chat_json(client, MODEL_NAME, prompt_male, temperature=0.5)
                save_result(response_male_text, file, rank, EXP_DIR, folder, try_no, "male")

                print(f"Processing {file} (Rank {rank}, Try {try_no}), Female...")
                prompt_female = insert_content(assessment_prompt, replace_name(candidate_profile, name_female), job_description)
                response_female_text = chat_json(client, MODEL_NAME, prompt_female, temperature=0.5)
                save_result(response_female_text, file, rank, EXP_DIR, folder, try_no, "female")